# SPER — Dubins + TSP (nearest neighbor)

Acest notebook folosește `dubins_planning.py` și motorul pure-Python `dubins_fgabbert_engine.py` (adaptat după [fgabbert/dubins_py](https://github.com/fgabbert/dubins_py)): traiectorii Dubins între `(x, y, θ)`, TSP nearest-neighbor, concatenare și figuri pentru `R_min ∈ {0.5, 1, 2, 5}`.

**Instalare (o dată):** din folderul `ProiectSPER` rulează `py -m pip install -r requirements.txt`, apoi deschide notebook-ul (ideal cu kernel din același mediu Python).

In [ ]:
# Cale robustă: Jupyter nu definește __file__; acceptă cwd = folderul `code` sau rădăcina proiectului
import os
import sys

_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, "dubins_planning.py")):
    HERE = _cwd
else:
    HERE = os.path.join(_cwd, "code")
if HERE not in sys.path:
    sys.path.insert(0, HERE)

from dubins_planning import (
    compare_nn_vs_random,
    concatenate_dubins_tour,
    get_dubins_path,
    plot_tour,
    run_experiment_radii,
    scenario_circular,
    scenario_few_points,
    scenario_many_points,
    tsp_nearest_neighbor,
)

import math

RADII = (0.5, 1.0, 2.0, 5.0)
OUT_DIR = os.path.normpath(os.path.join(HERE, "..", "output", "figuri"))
print("Figuri salvate în:", OUT_DIR)

## P2 — test `get_dubins_path` (2–3 perechi)
Unghiurile sunt în **radiani**.

In [ ]:
R = 1.0
tests = [
    ((0.0, 0.0, 0.0), (4.0, 0.0, math.pi / 2)),
    ((1.0, 1.0, math.pi / 4), (3.0, 0.0, -math.pi / 6)),
    ((0.0, 0.0, math.pi / 2), (2.0, 2.0, 0.0)),
]
for a, b in tests:
    pts = get_dubins_path(a, b, R, step_size=0.08)
    print("Start", a, "-> End", b, "| eșantioane:", len(pts))

### Observații după rularea testelor P2

La `R = 1.0` și `step_size = 0.08`, cele trei perechi au produs **61**, **93**, respectiv **39** de puncte `(x, y)` — numărul de eșantioane urmează aproximativ **lungimea geometrică Dubins / pas** (nu distanța euclidiană brută). Perechea a doua (de la `(1,1,π/4)` la `(3,0,-π/6)`) cere cele mai multe eșantioane: traiectoria trebuie să „întoarcă” mai mult vehiculul față de direcția țintă, deci domină arce și lungime totală mai mare. Prima pereche (capăt la `(4,0)` cu `θ = π/2`) e intermediară; a treia (`(0,0,π/2)` → `(2,2,0)`) e cea mai scurtă în lungime Dubins la același `R`, deci și polilinia e cea mai scurtă. Concluzie practică: **distanța în plan + diferența de orientare** controlează costul — unghiuri „nealiniate” cu coarda cresc atât lungimea traiectoriei, cât și numărul de eșantioane la pas fix.

## P3 + integrare — scenarii și figuri
- Puține puncte, multe puncte, aranjament circular  
- Bonus: compară lungimea traiectoriei **TSP NN** vs **ordine aleatoare** (aceleași puncte).

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

scenarios = {
    "scenariu_putine": scenario_few_points(),
    "scenariu_multe": scenario_many_points(12),
    "scenariu_circular": scenario_circular(10),
}

for name, pts in scenarios.items():
    run_experiment_radii(pts, RADII, OUT_DIR, base_name=name, use_nn=True)
    print("Salvat pentru", name)

    ln, lr = compare_nn_vs_random(pts, radius=1.0, seed=42)
    print(f"  Compară R=1.0: TSP NN = {ln:.3f} | random = {lr:.3f}")

print("Gata.")

### Observații după scenariile P3 (figuri + lungimi totale)

**Dependența de `R_min`:** pe același tur TSP-NN, lungimea totală crește monoton cu raza — exemplu scenariu „puține puncte”: **≈9.85** la `R=0.5`, **≈15.88** la `R=1`, **≈31.59** la `R=2`, **≈71.92** la `R=5`. Geometric, în reprezentarea normalizată la rază unitate unghiurile de arc rămân finite, iar re-scalarea la `R` mărește contribuția arcelor în metri; practic, **aceeași reorientare** pe cerc de rază mai mare înseamnă drum mai lung la viraje.

**Formă:** la `R=0.5` traiectoria e „zgomotoasă” / în zig-zag cu viraje strânse; la `R=5.0` arcele sunt largi, traseul e mai lin și ocolește mai mult spațiul între waypoint-uri.

**Multe vs puține puncte:** cu **12** puncte, lungimea la `R=1` a ieșit **≈54.9** față de **≈15.9** la setul mic — nu e doar numărul de segmente: fiecare concatenare fixează orientări intermediare, iar greedy-ul poate introduce **segmente Dubins lungi** dacă următorul vecin e aproape în plan dar „prost” din punct de vedere al tangentei.

**Cercul:** pe **10** puncte echidistante pe cerc, NN simplu poate „sări” pe cordă în loc să urmeze sensul circular; ordinea greedy nu optimizează structura 1D a variantei circulare — în figură se vede adesea un tur care **nu** parcurge monoton unghiul polar, ceea ce crește artificial lungimea Dubins.

### Observații după comparația TSP NN vs ordine aleatoare (`R=1`, `seed=42`)

Rezultatele tipărite de buclă **nu** arată că NN e întotdeauna mai bun: la scenariul **„puține puncte”** random e doar cu **≈0.59** mai lung (**≈16.46** vs **≈15.88**) — același set mic e tolerant la reordonări. La **„multe puncte”**, NN bate clar random: **≈54.86** vs **≈63.36** (**~8.5** diferență, ~**13.7%**), ceea ce ilustrează că ordinea contează când segmentele Dubins penalizează „salturile” proaste. Surpriza e la **„circular”**: NN **≈61.15** vs random **≈50.30** — random e **mai scurt cu ~10.9** (~**18%**), deci **o heuristică simplă + obiectiv Dubins non-euclidian** nu monotonizează față de permutarea aleatoare. Demonstrează că **TSP pe distanțe euclidiene ≠ optimizare directă a costului Dubins** și că un aranjament geometric regulat poate păcăli nearest-neighbor.